# Smoke tests para la operación Import Trips

Test / pruebas que sirven para validar que el contrato mínimo de la operación se respeta (emisión de issues, detalles, severidades) y asegurar que el control de flujo (raise) coincide con el catálogo (fatal/strict).

In [8]:
import sys
import os.path
sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd()))))))

from pylondrina.issues.core import emit_issue, emit_and_maybe_raise
from pylondrina.errors import SchemaError, ImportError as PylondrinaImportError

In [20]:
OP01_SMOKE_ISSUES: dict[str, IssueSpec] = {
    # 1) DataFrame vacío
    "IMP.INPUT.EMPTY_DATAFRAME": IssueSpec(
        code="IMP.INPUT.EMPTY_DATAFRAME",
        level="warning",
        fatal=False,
        exception="import",
        message_template="El DataFrame de entrada no contiene filas; se importará un dataset vacío.",
        details_keys=("rows_in", "columns_in", "note"),
        defaults={
            "rows_in": 0,
            "note": "empty_input",
        },
    ),

    # 3) TripSchema.fields vacío (fatal)
    "SCH.TRIP_SCHEMA.EMPTY_FIELDS": IssueSpec(
        code="SCH.TRIP_SCHEMA.EMPTY_FIELDS",
        level="error",
        fatal=True,
        exception="schema",
        message_template=(
            "El TripSchema no define campos (fields está vacío); "
            "no es posible importar con un esquema sin catálogo de campos."
        ),
        details_keys=("schema_version", "fields_size", "note"),
        defaults={
            "fields_size": 0,
            "note": "no_fields_defined",
        },
    ),

    # 8) Constraint no reconocida (regla desconocida) (fatal)
    "SCH.CONSTRAINTS.UNKNOWN_RULE": IssueSpec(
        code="SCH.CONSTRAINTS.UNKNOWN_RULE",
        level="error",
        fatal=True,
        exception="schema",
        message_template=(
            "El campo {field!r} incluye una constraint no soportada ({rule!r}); "
            "la validación no puede ejecutarse de forma consistente."
        ),
        details_keys=("field", "rule", "supported_rules", "action"),
        defaults={
            "action": "abort",
        },
    ),

    # 24) Colisión: múltiples canónicos apuntan a la misma columna fuente (fatal)
    "MAP.FIELDS.COLLISION_DUPLICATE_TARGET": IssueSpec(
        code="MAP.FIELDS.COLLISION_DUPLICATE_TARGET",
        level="error",
        fatal=True,
        exception="import",
        message_template="La correspondencia de campos produce colisión: múltiples canónicos apuntan a {source_column!r}.",
        details_keys=("source_column", "canonical_fields", "field_correspondence", "action"),
        defaults={
            "action": "abort",
        },
    ),

    # 17) strict_domains=False pero extendable=False por campo (warning)
    "DOM.POLICY.FIELD_NOT_EXTENDABLE": IssueSpec(
        code="DOM.POLICY.FIELD_NOT_EXTENDABLE",
        level="warning",
        fatal=False,
        exception="import",
        message_template=(
            "El campo {field!r} no admite extensión de dominio (extendable=False); "
            "los valores fuera de dominio no se incorporarán pese a que strict_domains=False."
        ),
        details_keys=("field", "strict_domains", "domain_extendable", "action"),
        defaults={
            "strict_domains": False,
            "domain_extendable": False,
            "action": "map_to_unknown",
        },
    ),

    # 30) strict_domains=True y fuera de dominio => abort (fatal)
    "DOM.STRICT.OUT_OF_DOMAIN_ABORT": IssueSpec(
        code="DOM.STRICT.OUT_OF_DOMAIN_ABORT",
        level="error",
        fatal=True,
        exception="import",
        message_template="Se detectaron valores fuera de dominio en {field!r} con strict_domains=True; import abortado.",
        details_keys=(
            "field",
            "unknown_count",
            "total_count",
            "unknown_rate",
            "unknown_examples",
            "policy",
            "action",
            "suggestion",
        ),
        defaults={
            "action": "abort",
        },
    ),

    # 31) Extensión de dominio aplicada (info)
    "DOM.EXTENSION.APPLIED": IssueSpec(
        code="DOM.EXTENSION.APPLIED",
        level="info",
        fatal=False,
        exception="import",
        message_template="Se extendió el dominio de {field!r} con {n_added} valores nuevos.",
        details_keys=(
            "field",
            "n_added",
            "added_values_sample",
            "added_values_total",
            "policy",
            "action",
        ),
        defaults={
            "action": "extended_domain",
        },
    ),

    # 27) Coerción mínima parcialmente fallida (warning)
    "IMP.TYPE.COERCE_PARTIAL": IssueSpec(
        code="IMP.TYPE.COERCE_PARTIAL",
        level="warning",
        fatal=False,
        exception="import",
        message_template=(
            "La conversión mínima del campo {field!r} falló en algunas filas ({fail_rate:.1%}); "
            "se marcarán como nulos para validación posterior."
        ),
        details_keys=(
            "field",
            "dtype_expected",
            "parse_fail_count",
            "total_count",
            "fail_rate",
            "fallback",
            "action",
        ),
        defaults={
            "fallback": "set_null",
            "action": "continue",
        },
    ),

    # Codigos pendientes
    "IMP.METADATA.DATASET_ID_CREATED": IssueSpec(
        code="IMP.METADATA.DATASET_ID_CREATED",
        level="info",
        fatal=False,
        exception="import",
        message_template="Se generó dataset_id para el dataset importado: {dataset_id!r}.",
        details_keys=("dataset_id", "generator", "stored_in"),
        defaults={
            "generator": "uuid4",
            "stored_in": ["TripDataset.metadata", "ImportReport.metadata"],
        },
    ),

    "IMP.INPUT.MISSING_REQUIRED_FIELD": IssueSpec(
        code="IMP.INPUT.MISSING_REQUIRED_FIELD",
        level="error",
        fatal=True,
        exception="import",
        message_template="Faltan campos obligatorios para importar según el TripSchema: {missing_required}.",
        details_keys=(
            "missing_required",
            "required",
            "source_columns",
            "field_correspondence_keys",
            "field_correspondence_values_sample",
        ),
    ),

    # Código SOLO para probar “error no fatal que depende de strict” (pendiente 2.3) ---
    # Nota: este code NO pertenece a OP-01; es un helper para validar el motor emit_and_maybe_raise.
    "TST.ERROR.STRICT_ONLY": IssueSpec(
        code="TST.ERROR.STRICT_ONLY",
        level="error",
        fatal=False,          # clave: si strict=False, NO debe lanzar
        exception="import",
        message_template="Error solo en strict: {reason}",
        details_keys=("reason",),
    ),
}

In [9]:
EXC_MAP_IMPORT = {
    "schema": SchemaError,
    "import": PylondrinaImportError,
}
DEFAULT_EXC = PylondrinaImportError

## Grupo A) Smoke tests WARNING/INFO (no lanzan): usan emit_issue

### Test A.1 Import con input vacío: warning sin abort

- Descripción: emite la issue cuando el DF no tiene filas; verifica que queda registrada con details mínimos.
- Objetivo: asegurar que el sistema registra el hallazgo pero no interrumpe el flujo (porque no es fatal).

In [10]:
issues = []
issue = emit_issue(
    issues, OP01_SMOKE_ISSUES, "IMP.INPUT.EMPTY_DATAFRAME",
    rows_in=0,
    columns_in=["a", "b"],
)
assert issue.code == "IMP.INPUT.EMPTY_DATAFRAME"
assert issue.level == "warning"
assert issue.details["rows_in"] == 0
assert issue.details["columns_in"] == ["a", "b"]
assert issue.details["note"] == "empty_input"
assert issue is issues[-1]

### Test A2. Dominio no extendible: warning + acción map_to_unknown

- Descripción: emite la issue para un campo categórico que no permite extensión real; valida action="map_to_unknown" y flags de política.
- Objetivo: asegurar que la política de “no extender” queda trazada en details y no aborta.

In [11]:
issues = []
issue = emit_issue(
    issues, OP01_SMOKE_ISSUES, "DOM.POLICY.FIELD_NOT_EXTENDABLE",
    field="mode",
    strict_domains=False,
    domain_extendable=False,
    action="map_to_unknown",
    row_count=12,
)
assert issue.code == "DOM.POLICY.FIELD_NOT_EXTENDABLE"
assert issue.level == "warning"
assert issue.field == "mode"
assert issue.row_count == 12
assert issue.details["strict_domains"] is False
assert issue.details["domain_extendable"] is False
assert issue.details["action"] == "map_to_unknown"

### Test A3. Extensión aplicada: info de auditoría 

- Descripción: emite la issue informativa cuando se extiende un dominio (cuando está permitido).
- Objetivo: verificar que este evento queda como info (auditoría), sin abortar.

In [12]:
issues = []
issue = emit_issue(
    issues, OP01_SMOKE_ISSUES, "DOM.EXTENSION.APPLIED",
    field="mode",
    n_added=1,
)
assert issue.code == "DOM.EXTENSION.APPLIED"
assert issue.level == "info"
assert issue.field == "mode"
assert issue.details["n_added"] == 1

### Test A4. Coerción parcial: warning con porcentaje formateado 

- Descripción: emite warning cuando la coerción deja nulos parciales; valida el mensaje con {fail_rate:.1%} (ej. “12.5%”).
- Objetivo: asegurar que la plantilla/formatting funciona y que el caso se registra como degradación no fatal.

In [13]:
issues = []
issue = emit_issue(
    issues, OP01_SMOKE_ISSUES, "IMP.TYPE.COERCE_PARTIAL",
    field="origin_time_utc",
    dtype_expected="datetime",
    parse_fail_count=3,
    total_count=24,
    fail_rate=3/24,   # 0.125 -> 12.5%
    fallback="set_null",
    action="continue",
)
assert issue.code == "IMP.TYPE.COERCE_PARTIAL"
assert issue.level == "warning"
assert issue.field == "origin_time_utc"
assert "12.5%" in issue.message  # por {fail_rate:.1%}
assert issue.details["parse_fail_count"] == 3
assert issue.details["total_count"] == 24
assert abs(issue.details["fail_rate"] - (3/24)) < 1e-12
assert issue.details["fallback"] == "set_null"
assert issue.details["action"] == "continue"

### Test A6. de warnings/infos (no deben lanzar)

In [21]:
issues = []

# warning no fatal
issue = emit_and_maybe_raise(
    issues, OP01_SMOKE_ISSUES, "IMP.INPUT.EMPTY_DATAFRAME",
    strict=False,
    exception_map=EXC_MAP_IMPORT,
    default_exception=DEFAULT_EXC,
    rows_in=0,
    columns_in=[],
)
assert issue.level == "warning"
assert len(issues) == 1

# info no fatal (requiere que agregues este code a tu catálogo)
issue2 = emit_and_maybe_raise(
    issues, OP01_SMOKE_ISSUES, "IMP.METADATA.DATASET_ID_CREATED",
    strict=False,
    exception_map=EXC_MAP_IMPORT,
    default_exception=DEFAULT_EXC,
    dataset_id="ds_001",
)
assert issue2.level == "info"
assert len(issues) == 2

## Grupo B) Smoke tests ERROR/FATAL (deben lanzar): usan emit_and_maybe_raise

### Test B1. Schema sin fields: fatal schema 

- Descripción: emite issue error y debe levantar SchemaError; valida que la issue gatillante se guardó antes del raise y que la excepción trae code/details/issues.
- Objetivo: comprobar el patrón clave: emitir evidencia + abortar.

In [14]:
issues = []
try:
    emit_and_maybe_raise(
        issues, OP01_SMOKE_ISSUES, "SCH.TRIP_SCHEMA.EMPTY_FIELDS",
        strict=False,
        exception_map=EXC_MAP_IMPORT,
        default_exception=DEFAULT_EXC,
        schema_version="1.1",
        fields_size=0,
    )
    assert False, "Debió lanzar SchemaError"
except SchemaError as e:
    assert e.code == "SCH.TRIP_SCHEMA.EMPTY_FIELDS"
    assert e.issue is issues[-1]              # issue emitida antes de abortar
    assert e.issue.level == "error"
    assert e.issue.details["schema_version"] == "1.1"
    assert e.issue.details["fields_size"] == 0
    assert e.issue.details["note"] == "no_fields_defined"
    assert len(e.issues) == 1                 # snapshot de issues acumuladas

### Test B2. Constraint desconocida: fatal schema

- Descripción: emite issue error y levanta SchemaError; valida placeholders {field!r} {rule!r} y details (supported_rules, action=abort).
- Objetivo: confirmar que los errores de schema con parámetros se convierten en excepción correctamente.

In [17]:
issues = []
try:
    emit_and_maybe_raise(
        issues, OP01_SMOKE_ISSUES, "SCH.CONSTRAINTS.UNKNOWN_RULE",
        strict=False,
        exception_map=EXC_MAP_IMPORT,
        default_exception=DEFAULT_EXC,
        field="origin_latitude",
        rule="h3",
        supported_rules=["nullable", "range", "pattern"],
        # action viene por default si lo definiste así en el spec
    )
    assert False, "Debió lanzar SchemaError"
except SchemaError as e:
    assert e.code == "SCH.CONSTRAINTS.UNKNOWN_RULE"
    assert e.issue is issues[-1]
    assert "origin_latitude" in e.issue.message
    assert "h3" in e.issue.message
    assert e.issue.details["field"] == "origin_latitude"
    assert e.issue.details["rule"] == "h3"
    assert isinstance(e.issue.details["supported_rules"], list)
    assert e.issue.details["action"] == "abort"

issues[-1].message

"El campo 'origin_latitude' incluye una constraint no soportada ('h3'); la validación no puede ejecutarse de forma consistente."

### Test B3. Colisión de mapping: fatal import 

- Descripción: emite issue error y levanta ImportError; valida details con source_column, canonical_fields, field_correspondence.
- Objetivo: asegurar que errores de mapping se clasifican como import y abortan.

In [18]:
issues = []
try:
    emit_and_maybe_raise(
        issues, OP01_SMOKE_ISSUES, "MAP.FIELDS.COLLISION_DUPLICATE_TARGET",
        strict=False,
        exception_map=EXC_MAP_IMPORT,
        default_exception=DEFAULT_EXC,
        source_column="lat_o",
        canonical_fields=["origin_latitude", "destination_latitude"],
        field_correspondence={"origin_latitude": "lat_o", "destination_latitude": "lat_o"},
    )
    assert False, "Debió lanzar ImportError"
except PylondrinaImportError as e:
    assert e.code == "MAP.FIELDS.COLLISION_DUPLICATE_TARGET"
    assert e.issue is issues[-1]
    assert e.issue.details["source_column"] == "lat_o"
    assert isinstance(e.issue.details["canonical_fields"], list)
    assert isinstance(e.issue.details["field_correspondence"], dict)
    assert e.issue.details["action"] == "abort"

### Test B4. Strict domains + fuera de dominio: fatal import 

- Descripción: emite issue error y levanta ImportError; valida policy.strict_domains=True, action=abort.
- Objetivo: asegurar que la política de dominios (fatalidad) funciona y que los details relevantes quedan pegados al error.

In [19]:
issues = []
try:
    emit_and_maybe_raise(
        issues, OP01_SMOKE_ISSUES, "DOM.STRICT.OUT_OF_DOMAIN_ABORT",
        strict=False,
        exception_map=EXC_MAP_IMPORT,
        default_exception=DEFAULT_EXC,
        field="mode",
        unknown_count=5,
        total_count=100,
        unknown_rate=0.05,
        unknown_examples=["taxi", "moto"],
        policy={"strict_domains": True, "domain_extendable": True},
        action="abort",
    )
    assert False, "Debió lanzar ImportError"
except PylondrinaImportError as e:
    assert e.code == "DOM.STRICT.OUT_OF_DOMAIN_ABORT"
    assert e.issue is issues[-1]
    assert e.issue.field == "mode"
    assert e.issue.details["action"] == "abort"
    assert e.issue.details["policy"]["strict_domains"] is True
    assert len(e.issues) == 1

### Test B6. Smoke test de fatales (deben lanzar siempre)

In [23]:
# fatal schema
issues = []
try:
    emit_and_maybe_raise(
        issues, OP01_SMOKE_ISSUES, "SCH.TRIP_SCHEMA.EMPTY_FIELDS",
        strict=False,
        exception_map=EXC_MAP_IMPORT,
        default_exception=DEFAULT_EXC,
        schema_version="1.1", fields_size=0
    )
    assert False
except SchemaError as e:
    assert e.code == "SCH.TRIP_SCHEMA.EMPTY_FIELDS"
    assert len(e.issues) == 1

# fatal import: missing required (requiere agregar este code a tu catálogo)
issues = []
try:
    emit_and_maybe_raise(
        issues, OP01_SMOKE_ISSUES, "IMP.INPUT.MISSING_REQUIRED_FIELD",
        strict=False,
        exception_map=EXC_MAP_IMPORT,
        default_exception=DEFAULT_EXC,
        missing_required=["user_id", "origin_latitude"],
        action="abort"
    )
    assert False
except PylondrinaImportError as e:
    assert e.code == "IMP.INPUT.MISSING_REQUIRED_FIELD"
    assert e.issue is issues[-1]

## Smoke test “error solo strict”

En OP-01 import, casi todo level="error" es fatal. Pero el sistema sí soporta el caso “error no fatal, que solo aborta si strict=True”. Este smoke test valida esa capacidad, usando un code dummy (de pruebas), porque OP-01 no suele tener ese patrón.

1. Definir en un catálogo de test (puede ser DUMMY_ERR_ISSUES) un IssueSpec:
    - level="error"
    - fatal=False
    - exception="import"
2. Probar:
    - strict=False → no lanza
    - strict=True → lanza

In [24]:
# strict=False -> NO lanza
issues = []
issue = emit_and_maybe_raise(
    issues, OP01_SMOKE_ISSUES, "TST.ERROR.STRICT_ONLY",
    strict=False,
    exception_map=EXC_MAP_IMPORT,
    default_exception=DEFAULT_EXC,
    reason="algo malo pero tolerable"
)
assert issue.level == "error"
assert len(issues) == 1

# strict=True -> SÍ lanza
issues = []
try:
    emit_and_maybe_raise(
        issues, OP01_SMOKE_ISSUES, "TST.ERROR.STRICT_ONLY",
        strict=True,
        exception_map=EXC_MAP_IMPORT,
        default_exception=DEFAULT_EXC,
        reason="algo malo pero tolerable"
    )
    assert False
except PylondrinaImportError as e:
    assert e.code == "TST.ERROR.STRICT_ONLY"
    assert e.details["reason"] == "algo malo pero tolerable"
    assert len(e.issues) == 1